# 02 · Temporal reconstruction

The board stays still, the lecturer does not. For every pixel of the rectified board the reconstruction keeps the **median of the
last N samples in which it was not covered**. This yields:

- the clean board at every moment (a snapshot when the content changes, that is writing or erasing);
- the spot pointed at by the lecturer in board coordinates, with a timestamp, to align with the transcript.

Without a real video a synthetic sequence is used: background derived from the frame (inpainting of the lecturer)
and a silhouette moving in front of the board. With a real video the CLI is enough:

```
vision-toolkit board data/video/lecture.mp4 --step 1 --crop 0,68,1674,783
```

In [ ]:
from pathlib import Path
import cv2, numpy as np, matplotlib.pyplot as plt
from vision_toolkit.core.geometry import rectify_homography
from vision_toolkit.core.temporal import PlaneMedian
from vision_toolkit.usecases.board.detect import detect_board
from vision_toolkit.usecases.board.person_mask import PersonMasker

ROOT = Path.cwd().parent
x, y, w, h = 0, 68, 1674, 783
frame = cv2.imread(str(ROOT / "data/frame/sample_censored.png"))[y:y+h, x:x+w]
masker = PersonMasker()
person = masker(frame)
background = cv2.inpaint(frame, person.mask, 7, cv2.INPAINT_TELEA)

def paste(dx):
    M = np.float32([[1, 0, dx], [0, 1, 0]])
    moved = cv2.warpAffine(frame, M, (w, h))
    m = cv2.warpAffine(cv2.erode(person.mask, np.ones((15, 15), np.uint8)), M, (w, h)) > 0
    out = background.copy(); out[m] = moved[m]; return out

offsets = [0, 250, 500, 750, 1000, 750, 500, 250, 0, -250]
frames = [paste(dx) for dx in offsets]
quad = detect_board(frame, person.mask)
H, size = rectify_homography(quad)

In [ ]:
rec = PlaneMedian(H, size, window=5)
track = []
for f in frames:
    p = masker(f)
    rec.update(f, p.mask)
    track.append(rec.to_plane(p.pointer))
masker.close()

truth = cv2.warpPerspective(background, H, size).astype(np.float32)
err = lambda img: float((np.abs(img.astype(np.float32) - truth).max(axis=2) > 40).mean())
print(f"wrong pixels: single frame {err(cv2.warpPerspective(frames[-1], H, size)):.4f}, reconstruction {err(rec.background):.4f}")

fig, axs = plt.subplots(2, 1, figsize=(14, 8))
axs[0].imshow(cv2.cvtColor(cv2.warpPerspective(frames[-1], H, size), cv2.COLOR_BGR2RGB)); axs[0].set_title("single frame")
axs[1].imshow(cv2.cvtColor(rec.image(), cv2.COLOR_BGR2RGB)); axs[1].set_title("reconstruction + pointed spots")
pts = np.array([t for t in track if t is not None])
axs[1].plot(pts[:, 0], pts[:, 1], "o-", color="lime")
for a in axs: a.axis("off")
plt.tight_layout(); plt.show()

## To check on a real video

- **Shadows** of the lecturer on the board: they are not in the person mask.
- **Writing** happens under the hand, so it shows up in the reconstruction only when the hand moves away (delay about window × step).
- **Snapshot threshold** (`--min-change`): tune it on a whole lecture.
- **Pointed spot**: tell pointing from gesturing (`arm_angle`, how long the pose lasts).